# HaRP — Full Pipeline on Colab GPU (Exp 02 → 16)

Runs all experiments using the existing pipeline scripts unchanged.
Ollama handles Exp 02, 03, 14. HuggingFace GPU handles Exp 05, 08, 11.

## Before running:
1. **T4 GPU**: `Runtime → Change runtime type → T4 GPU → Save`
2. **Upload zip**: Files panel (folder icon) → Upload → `harp_colab_upload.zip`
3. Run all cells top-to-bottom.

## Time estimate:
| Phase | Experiments | Time |
|-------|-------------|------|
| Setup (deps + Ollama + model pull) | — | ~10 min |
| Consistency signals (checkpointed) | 02 | ~60 min |
| Labeling | 03 | ~20 min |
| V1 estimator | 04 | <1 min |
| Hidden states (GPU) | 05 | ~20 min |
| Probing + geometry | 06–07 | ~3 min |
| Logit lens (GPU) | 08 | ~15 min |
| Features + steering vector | 09–10 | ~2 min |
| Activation steering (GPU) | 11 | ~15 min |
| Policy + baselines + calibration + eval | 12–16 | ~5 min |
| **Total** | | **~2.5 hours** |

## After running:
Last cell downloads `harp_colab_results.zip`. Extract it into your local `harp/` folder.

In [ ]:
# CELL 1 — Verify GPU runtime
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'NO GPU detected!\n'
        'Go to Runtime → Change runtime type → T4 GPU → Save, then reconnect.'
    )

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'PyTorch: {torch.__version__}')
print('GPU check passed — ready to proceed.')

In [ ]:
# CELL 2 — Extract uploaded zip and set working directory
import zipfile, os, sys
from pathlib import Path

zip_path = '/content/harp_colab_upload.zip'
if not Path(zip_path).exists():
    raise FileNotFoundError(
        'harp_colab_upload.zip not found!\n'
        'Upload it first: Files panel (folder icon) → Upload → select the zip.'
    )

# The zip has no top-level prefix, so we create harp/ and extract into it
ROOT = Path('/content/harp')
ROOT.mkdir(parents=True, exist_ok=True)

print('Extracting harp_colab_upload.zip ...')
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(str(ROOT))
print('Extraction complete.')

print('\nChecking extracted contents:')
for d in ['src', 'experiments', 'data/processed']:
    p = ROOT / d
    status = 'OK' if p.exists() else 'MISSING — recheck zip'
    print(f'  {d}: {status}')

# Create output directories
for d in ['results/logs', 'results/plots', 'results/hidden_states']:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

# Set working directory — persists for all cells in this session
os.chdir(str(ROOT))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'\nWorking directory set to: {os.getcwd()}')

In [ ]:
# CELL 3 — Install Python dependencies
%pip install -q transformers==4.48.0 accelerate sentence-transformers scipy scikit-learn tqdm datasets
print('Python packages installed.')

---
## Install Ollama + Pull Qwen Model
Ollama is required for Exp 02 (consistency signals), Exp 03 (labeling), and Exp 14 (sycophancy).

In [ ]:
# CELL 4 — Install Ollama binary (no systemd needed)
import subprocess, json, time, os

# Step 1: Install zstd (Ollama releases are .tar.zst format)
print('[1/5] Installing zstd decompressor...')
subprocess.run('apt-get install -qq -y zstd', shell=True, check=True)
print('      zstd ready.')

# Step 2: Get latest release info from GitHub API
print('[2/5] Fetching Ollama release info from GitHub API...')
r = subprocess.run(
    'curl -s https://api.github.com/repos/ollama/ollama/releases/latest',
    shell=True, capture_output=True, text=True
)
release = json.loads(r.stdout)
version = release['tag_name']
assets  = {a['name']: a['browser_download_url'] for a in release.get('assets', [])}
print(f'      Version: {version}')

# Step 3: Download ollama-linux-amd64.tar.zst
url = assets.get('ollama-linux-amd64.tar.zst')
if not url:
    raise RuntimeError(f'ollama-linux-amd64.tar.zst not found. Available: {list(assets.keys())}')
print(f'[3/5] Downloading {url} ...')
subprocess.run(
    f'curl -fsSL "{url}" -o /tmp/ollama-linux-amd64.tar.zst',
    shell=True, check=True
)
print('      Download complete.')

# Step 4: Extract into /usr/local/ (puts bin/ollama in place)
print('[4/5] Extracting archive...')
subprocess.run(
    'tar --use-compress-program=zstd -xf /tmp/ollama-linux-amd64.tar.zst -C /usr/local/',
    shell=True, check=True
)
# Binary may land at /usr/local/bin/ollama or /usr/local/ollama depending on archive layout
for candidate in ['/usr/local/bin/ollama', '/usr/local/ollama']:
    if os.path.exists(candidate):
        if candidate != '/usr/local/bin/ollama':
            subprocess.run(f'mv "{candidate}" /usr/local/bin/ollama', shell=True, check=True)
        break
subprocess.run('chmod +x /usr/local/bin/ollama', shell=True, check=True)

# Verify it's a real ELF binary (not an HTML error page)
with open('/usr/local/bin/ollama', 'rb') as f:
    magic = f.read(4)
if magic != b'\x7fELF':
    raise RuntimeError(f'Binary is not a valid ELF file. Got magic bytes: {magic!r}')
print('      Binary verified (valid ELF).')

# Step 5: Start Ollama server in background
print('[5/5] Starting Ollama server...')
subprocess.Popen(
    ['/usr/local/bin/ollama', 'serve'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(10)

# Verify server is reachable
for attempt in range(6):
    try:
        result = subprocess.run(
            'curl -s http://localhost:11434',
            shell=True, capture_output=True, timeout=5
        )
        if result.returncode == 0:
            print('Ollama server: running')
            break
    except Exception:
        pass
    print(f'  Waiting for server ({attempt+1}/6)...')
    time.sleep(5)
else:
    print('WARNING: Server may not be ready — continuing anyway.')

In [ ]:
# CELL 5 — Pull Qwen 2.5 3B (~2 GB, 3-5 min)
import subprocess
print('Pulling qwen2.5:3b ...')
subprocess.run(['/usr/local/bin/ollama', 'pull', 'qwen2.5:3b'], check=True)
print('Model ready.')

---
## Experiments 02–04 — Consistency Signals, Labeling, V1 Estimator

All three use the existing pipeline scripts unchanged.
**Exp 02 is checkpointed** — it skips any `prompt_id`s already scored in previous runs.

In [ ]:
# CELL 6 — Exp 02: Consistency Signals (~60 min, checkpointed)
import os; os.chdir('/content/harp')
print('=' * 60)
print('Experiment 02 — Consistency Signals (Group B)')
print('=' * 60)
%run experiments/02_consistency_check.py

In [ ]:
# CELL 7 — Exp 03: Label Responses (~20 min)
import os; os.chdir('/content/harp')
print('=' * 60)
print('Experiment 03 — Label Responses (LLM-as-judge)')
print('=' * 60)
%run experiments/03_label_responses.py

In [ ]:
# CELL 8 — Exp 04: Train V1 Estimator (~seconds)
import os; os.chdir('/content/harp')
print('=' * 60)
print('Experiment 04 — Train Estimator V1 (Group A only)')
print('=' * 60)
%run experiments/04_train_estimator_v1.py

---
## Experiment 05 — Extract Hidden States (GPU)

Loads Qwen 2.5 3B via HuggingFace Transformers and runs 700 forward passes.
Extracts last-token hidden states at every transformer layer for downstream probing and geometry analysis.
This is the core GPU step (~20 min on T4).

In [ ]:
# CELL 9 — Exp 05: Extract Hidden States (~20 min on T4 GPU)
import os; os.chdir('/content/harp')
print('=' * 60)
print('Experiment 05 — Hidden State Extraction (GPU)')
print('=' * 60)
%run experiments/05_extract_hidden_states.py

---
## Experiments 06–07 — Layer Probing & Geometry of Truth (CPU, fast)

In [ ]:
# CELL 10 — Exp 06: Probe Layers (~1-2 min)
import os; os.chdir('/content/harp')
print('=' * 60)
print('Experiment 06 — Layer Probing')
print('=' * 60)
%run experiments/06_probe_layers.py

In [ ]:
# CELL 11 — Exp 07: Geometry of Truth (~1-2 min)
import os; os.chdir('/content/harp')
print('=' * 60)
print('Experiment 07 — Geometry of Truth')
print('=' * 60)
%run experiments/07_geometry_of_truth.py

---
## Experiment 08 — Logit Lens (GPU)

Projects intermediate hidden states through the unembedding matrix to track how token predictions
evolve across layers. Uses GPU (~10-15 min).

In [ ]:
# CELL 12 — Exp 08: Logit Lens (~10-15 min on GPU)
import os; os.chdir('/content/harp')
print('=' * 60)
print('Experiment 08 — Logit Lens')
print('=' * 60)
%run experiments/08_logit_lens.py

---
## Experiments 09–10 — Feature Assembly & Steering Vector (CPU)

In [ ]:
# CELL 13 — Exp 09: Train V2 Estimator (all features combined, ~1 min)
import os; os.chdir('/content/harp')
print('=' * 60)
print('Experiment 09 — Feature Combination (V2 Estimator)')
print('=' * 60)
%run experiments/09_train_estimator_v2.py

In [ ]:
# CELL 14 — Exp 10: Compute Steering Vector (~1 min)
import os; os.chdir('/content/harp')
print('=' * 60)
print('Experiment 10 — Steering Vector Computation')
print('=' * 60)
%run experiments/10_compute_steering_vector.py

---
## Experiment 11 — Activation Steering (GPU)

Applies the computed steering vector to the model at inference time and measures the effect
on truthfulness. Uses GPU (~10-20 min).

In [ ]:
# CELL 15 — Exp 11: Activation Steering (~10-20 min on GPU)
import os; os.chdir('/content/harp')
print('=' * 60)
print('Experiment 11 — Activation Steering')
print('=' * 60)
%run experiments/11_run_steering_experiment.py

---
## Experiments 12–16 — Policy, Baselines, Sycophancy, Calibration, Full Eval (CPU)

All sklearn/numpy — fast. Exp 14 uses Ollama (already running from Cell 4).

In [ ]:
# CELL 16 — Exp 12: Authorization Policy (LAMBDA=0.2, ~1 min)
import os; os.chdir('/content/harp')
print('=' * 60)
print('Experiment 12 — Authorization Policy (LAMBDA=0.2)')
print('=' * 60)
%run experiments/12_run_policy.py

In [ ]:
# CELL 17 — Exp 13: Baseline Comparison (~1 min)
import os; os.chdir('/content/harp')
print('=' * 60)
print('Experiment 13 — Baseline Comparison')
print('=' * 60)
%run experiments/13_run_baselines.py

In [ ]:
# CELL 18 — Exp 14: Sycophancy Analysis (~5-10 min, uses Ollama)
import os; os.chdir('/content/harp')
print('=' * 60)
print('Experiment 14 — Sycophancy Analysis')
print('=' * 60)
%run experiments/14_sycophancy_analysis.py

In [ ]:
# CELL 19 — Exp 15: Calibration / ECE (~1 min)
import os; os.chdir('/content/harp')
print('=' * 60)
print('Experiment 15 — Calibration (ECE)')
print('=' * 60)
%run experiments/15_calibration.py

In [ ]:
# CELL 20 — Exp 16: Full Evaluation (aggregates all results, ~1 min)
import os; os.chdir('/content/harp')
print('=' * 60)
print('Experiment 16 — Full Evaluation')
print('=' * 60)
%run experiments/16_full_evaluation.py

---
## Package & Download Results

Packs all outputs into `harp_colab_results.zip` and downloads it.
Extract it into your local `harp/` folder, then run `streamlit run app.py`.

In [ ]:
# CELL 21 — Package all results into a zip
import zipfile, os
from pathlib import Path

output_zip = '/content/harp_colab_results.zip'
root = Path('/content/harp')

include_patterns = [
    'results/hidden_states/*.npz',
    'results/logs/*.json',
    'results/plots/*.png',
    'data/processed/features_combined.csv',
    'data/processed/features_groupAB.csv',
    'data/processed/truthfulqa_labeled.csv',
    'data/processed/nq_labeled.csv',
    'data/processed/adversarial_labeled.csv',
    'data/processed/features_semantic_entropy.csv',
]

files_added = []
with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for pattern in include_patterns:
        for f in sorted(root.glob(pattern)):
            arcname = str(f.relative_to(root))
            zf.write(f, arcname)
            files_added.append(arcname)

size_mb = os.path.getsize(output_zip) / (1024 ** 2)
print(f'Packed {len(files_added)} files ({size_mb:.1f} MB):')
for f in sorted(files_added):
    print(f'  {f}')

In [ ]:
# CELL 22 — Download results zip
from google.colab import files
files.download('/content/harp_colab_results.zip')
print('Download started.')
print()
print('On your local machine, extract with:')
print('  cd ~/Desktop/harp && unzip -o ~/Downloads/harp_colab_results.zip')
print()
print('Then run:')
print('  bash run_local_phase3.sh   # Exp 12-16 on local CPU (optional re-run)')
print('  streamlit run app.py       # Launch the app')